# Week 10 Lab 2: GPU Accelerated Inference (CUDA)

**Goal:** Run Llama 3.1 8B on a machine with an NVIDIA GPU for high-speed inference.

## 1. Why GPU Inference?
Large Language Models represent data as matrices. GPUs are specialized for matrix multiplication. Running a model on a GPU (via CUDA) is typically **20x to 100x faster** than on a CPU. However, you are limited by **VRAM** (Video RAM).

**Prerequisites:** NVIDIA GPU with 6GB+ VRAM & CUDA installed.

## 2. Install Dependencies
We need `transformers`, `accelerate`, and `bitsandbytes` (for 8-bit loading).

In [ ]:
!pip install torch transformers accelerate bitsandbytes

## 3. Load the Model
We will load **Meta-Llama-3.1-8B-Instruct** directly from Hugging Face.
*   `load_in_8bit=True` drastically reduces VRAM usage.
*   `device_map="auto"` automatically assigns the model to your GPU.

In [ ]:
import torch
from transformers import pipeline

# Ensure you are logged in or allow access if prompt appears
# from huggingface_hub import login
# login()

MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

print(f"Loading {MODEL_ID} onto GPU... (This requires logging into Hugging Face)")

try:
    pipe = pipeline(
        "text-generation",
        model=MODEL_ID,
        model_kwargs={"torch_dtype": torch.bfloat16, "load_in_8bit": True},
        device_map="auto",
    )
    print("Model Loaded on GPU!")
except Exception as e:
    print(f"Error loading model: {e}\nEnsure you have accepted the model terms on Hugging Face and have a valid token.")

## 4. Chat with the AI
Run the cell below to chat. Notice the speed capabilities of the GPU!

In [ ]:
def chat_with_gpu_model():
    print("Starting GPU Chat... (Type 'exit' to quit)")
    while True:
        user_input = input("You: ")
        if user_input.lower() in ["exit", "quit"]:
            break
        
        # Llama 3 Prompt Format
        messages = [
            {"role": "user", "content": user_input},
        ]
        
        # Run Inference
        outputs = pipe(
            messages,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
        
        print(f"\nAI: {outputs[0]['generated_text'][-1]['content']}")
        print("-" * 50)

if 'pipe' in locals():
    chat_with_gpu_model()